In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - 분류: Multiverse Computing의 Qiskit Function
> **Note:** * Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, 및 On-Prem(IBM Quantum Platform API를 통한) Plan 사용자만 사용할 수 있는 실험적 기능입니다. 미리보기 릴리스 상태이며 변경될 수 있습니다.

## 개요
"Singularity Machine Learning - Classification" 함수를 사용하면, 양자 전문 지식 없이도 양자 하드웨어에서 실제 머신러닝 문제를 해결할 수 있습니다. 앙상블 방법 기반의 이 애플리케이션 함수는 하이브리드 분류기입니다. 초기 앙상블 훈련을 위해 부스팅, 배깅, 스태킹과 같은 고전적 방법을 활용합니다. 이후 변분 양자 고유값 솔버(VQE) 및 양자 근사 최적화 알고리즘(QAOA)과 같은 양자 알고리즘을 사용하여 훈련된 앙상블의 다양성, 일반화 능력, 전체 복잡성을 향상시킵니다.

다른 양자 머신러닝 솔루션과 달리, 이 함수는 대상 QPU의 Qubit 수에 제한받지 않고 수백만 개의 예제와 특성을 가진 대규모 데이터셋을 처리할 수 있습니다. Qubit 수는 훈련할 수 있는 앙상블의 크기만 결정합니다. 또한 매우 유연하여, 금융, 의료, 사이버 보안을 포함한 광범위한 도메인의 분류 문제를 해결하는 데 사용할 수 있습니다.
고차원, 노이즈, 불균형 데이터셋과 관련된 고전적으로 어려운 문제에서 일관되게 높은 정확도를 달성합니다.
![작동 방식](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
다음과 같은 사용자를 위해 구축되었습니다:
1. 양자 머신러닝을 제품 및 서비스에 통합하여 기술 제공물을 향상시키려는 기업의 엔지니어와 데이터 과학자,
2. 양자 머신러닝 응용 프로그램을 탐색하고 분류 작업에 양자 컴퓨팅을 활용하려는 양자 연구소의 연구자, 그리고
3. 머신러닝과 같은 과정의 교육 기관에서 양자 컴퓨팅의 장점을 시연하려는 학생과 교사.

다음 예제는 `create`, `list`, `fit`, `predict`를 포함한 다양한 기능을 보여주며, 비선형 결정 경계로 인해 악명 높게 어려운 문제인 두 개의 인터리빙 반원으로 구성된 합성 문제에서의 사용법을 시연합니다.


## 함수 설명
이 Qiskit Function을 사용하면 Singularity의 양자 향상 앙상블 분류기를 사용하여 이진 분류 문제를 해결할 수 있습니다. 내부적으로, 레이블된 데이터셋에 대해 고전적으로 분류기 앙상블을 훈련한 다음, IBM&reg; QPU에서 양자 근사 최적화 알고리즘(QAOA)을 사용하여 최대 다양성과 일반화를 위해 최적화하는 하이브리드 접근 방식을 사용합니다. 사용자 친화적인 인터페이스를 통해 요구 사항에 따라 분류기를 구성하고, 선택한 데이터셋에 대해 훈련하며, 이전에 보지 못한 데이터셋에 대해 예측할 수 있습니다.

일반적인 분류 문제를 해결하려면:
1. 데이터셋을 전처리하고, 훈련 세트와 테스트 세트로 분할합니다. 선택적으로, 훈련 세트를 훈련 세트와 검증 세트로 추가 분할할 수 있습니다. [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html)을 사용하여 이를 수행할 수 있습니다.
2. 훈련 세트가 불균형한 경우, [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets)을 사용하여 클래스의 균형을 맞추기 위해 리샘플링할 수 있습니다.
3. 카탈로그의 `file_upload` 메서드를 사용하여 훈련, 검증, 테스트 세트를 함수의 스토리지에 개별적으로 업로드하고, 매번 관련 경로를 전달합니다.
4. 함수의 `create` 액션을 사용하여 양자 분류기를 초기화합니다. 학습기의 수와 유형, 정규화(람다 값), 레이어 수, 고전적 옵티마이저 유형, 양자 Backend 등을 포함한 최적화 옵션과 같은 하이퍼파라미터를 허용합니다.
5. 함수의 `fit` 액션을 사용하여 레이블된 훈련 세트(해당하는 경우 검증 세트 포함)를 전달하여 훈련 세트에서 양자 분류기를 훈련합니다.
6. 함수의 `predict` 액션을 사용하여 이전에 보지 못한 테스트 세트에 대해 예측합니다.
## 액션 기반 접근 방식
이 함수는 액션 기반 접근 방식을 사용합니다. 액션을 사용하여 작업을 수행하거나 상태를 변경하는 가상 환경이라고 생각할 수 있습니다. 현재 다음 액션을 제공합니다: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict), [create_fit_predict](#7-create-fit-predict). 다음 예제는 `create_fit_predict` 액션을 시연합니다.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List
`list` 액션은 공유 데이터 디렉터리에서 `*.pkl.tar` 형식의 저장된 모든 분류기를 검색합니다. `catalog.files()` 메서드를 사용하여 이 디렉터리의 내용에 접근할 수도 있습니다. 일반적으로 list 액션은 공유 데이터 디렉터리에서 `*.pkl.tar` 확장자를 가진 파일을 검색하고 목록 형식으로 반환합니다.
#### 입력
|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### 사용법

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create
`create` 액션은 제공된 매개변수를 사용하여 지정된 `quantum_classifier` 유형의 분류기를 생성하고, 공유 데이터 디렉터리에 저장합니다.

> **Note:** 현재 이 함수는 `QuantumEnhancedEnsembleClassifier`만 지원합니다.
#### 입력
|     Name    |    Type     | Description | Required  | Default |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes | - |
| `name` | `str` | The name of the quantum classifier, e.g., `spam_classifier`. | Yes | - |
| `instance` | `str` | IBM instance. | Yes | - |
| `backend_name` | `str` | IBM compute resource. Default is `None`, which means the backend with the fewest pending jobs will be used. | No | `None` |
| `quantum_classifier` | `str` | The type of the quantum classifier, i.e., `QuantumEnhancedEnsembleClassifier`. | No | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | The number of learners in the ensemble. | No | `10` |
| `learners_types` | `list` | Types of learners. Among supported types are: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier`, and `LogisticRegression`. Further details related to each can be found in the [scikit-learn documentation](https://scikit-learn.org/stable/supervised_learning.html). | No | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proportions of each learner type in the ensemble. | No | `[1.0]` |
| `learners_options` | `list` | Options for each learner type in the ensemble. For a complete list of options corresponding to the chosen learner type/s, consult [scikit-learn documentation](https://scikit-learn.org/stable/supervised_learning.html). | No | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` or `list` | Type/s of regularization to be used: `onsite` or `alpha`. `onsite` controls the onsite term where higher values lead to sparser ensembles. `alpha` controls trade-off between interaction and onsite terms where lower values lead to sparser ensembles. If a list is provided, models will be trained for each type and the best performing one will be selected. | No | `onsite` |
| `regularization` | `str` or `float` or `list` | Regularization value. Bounded between `0` and `+inf` if regularization_type is `onsite`. Bounded between `0` and `1` if regularization_type is `alpha`. If set to `auto`, auto-regularization is used - optimal regularization parameter is found by binary search with the desired ratio of selected classifiers to total classifiers (`regularization_desired_ratio`) and the upper bound for the regularization parameter (`regularization_upper_bound`). If a list is provided, models will be trained for each value and the best performing one will be selected. | No | `0.01` |
| `regularization_desired_ratio` | `float` or `list` | Desired ratio/s of selected classifiers to total classifiers for auto-regularization. If a list is provided, models will be trained for each ratio and the best performing one will be selected. | No | `0.75` |
| `regularization_upper_bound` | `float` or `list` | Upper bound/s for the regularization parameter when using auto-regularization. If a list is provided, models will be trained for each upper bound and the best performing one will be selected. | No | `200` |
| `weight_update_method` | `str` | Method for update of sample weights from among `logarithmic` and `quadratic`. | No | `logarithmic` |
| `sample_scaling` | `boolean` | Whether sample scaling should be applied. | No | `False` |
| `prediction_scaling` | `float` | Scaling factor for predictions. | No | `None` |
| `optimizer_options` | `dictionary` | QAOA optimizer options. A list of available options is presented later in this documentation. | No | ... |
| `voting` | `str` | Use majority voting (`hard`) or average of probabilities (`soft`) for aggregating learners' predictions/probabilities. | No | `hard` |
| `prob_threshold` | `float` | Optimal probability threshold. | No | `0.5` |
| `random_state` | `integer` | Control randomness for repeatability. | No | `None` |


- 추가로, `optimizer_options`는 다음과 같습니다:

|     Name    |    Type     | Description | Required  | Default |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | The number of solutions | No | `1024` |
| `reps` | `integer` | The number of repetitions | No | `4` |
| `sparsify` | `float` | The sparsification threshold | No | `0.001` |
| `theta` | `float` | The initial value of theta, a variational parameter of QAOA | No | `None` |
| `simulator` | `boolean` | Whether to use a simulator or a QPU | No | `False` |
| `classical_optimizer` | `str` | Name of the classical optimizer for the QAOA. All solvers offered by SciPy, as enlisted [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), are usable. You will need to set `classical_optimizer_options` accordingly | No | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Classical optimizer options. For a complete list of available options, consult [SciPy documentation](https://docs.scipy.org/doc/scipy/reference/) | No | `{"maxiter": 60}` |
| `optimization_level` | `integer` | The depth of the QAOA circuit | No | `3` |
| `num_transpiler_runs` | `integer` | Number of transpiler runs | No | `30` |
| `pass_manager_options` | `dictionary` | Options for generating preset pass manager | No | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Estimator options. For a complete list of available options, consult [Qiskit Runtime Client documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | No | `None` |
| `sampler_options` | `dictionary` | Sampler options. For a complete list of available options, consult the [Qiskit Runtime Client documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | No | `None` |

- 기본 `estimator_options`는 다음과 같습니다:

|     Name    |    Type     | Value  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- 기본 `sampler_options`는 다음과 같습니다:

|     Name    |    Type     | Value |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### 사용법

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### 검증
- `name`:
    - 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - 공유 데이터 디렉터리에 같은 이름의 분류기가 이미 존재해야 합니다.
### 4. Fit
`fit` 액션은 제공된 훈련 데이터를 사용하여 분류기를 훈련합니다.

#### 입력
|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | The name of the action. Must be `fit`. | Yes |
| `name`      | `str`    | The name of the classifier to train. | Yes |
| `X`         | `array` or `list` or `str` | The training data. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `y`         | `array` or `list` or `str` | The training target values. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `fit_params`| `dictionary`| Additional parameters to pass to the `fit` method of the classifier. | No |

##### fit_params
|     Name    |    Type     | Description |   Required  |   Default   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | The validation data and labels. | No | `None` |
| `pos_label` | `integer` or `str` | The class label to be mapped to 1. | No | `None` |
| `optimization_data` | `str` | Dataset to optimize the ensemble on. Can be one of: `train`, `validation`, `both`. | No | `train` |

#### 사용법

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### 검증
- `name`:
    - 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - 공유 데이터 디렉터리에 같은 이름의 분류기가 이미 존재해야 합니다.
### 5. Predict
`predict` 액션은 경성 및 연성 예측(확률)을 얻는 데 사용됩니다.

#### 입력
|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | The name of the action. Must be `predict`. | Yes |
| `name`      | `str`    | The name of the classifier to be used. | Yes |
| `X`         | `array` or `list` or `str` | The test data. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `options["out"]` | `str` | The output JSON filename to save the predictions in the shared data directory. If not provided, the predictions are returned in the job result. | No |

#### 사용법

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### 검증
- `name`:
    - 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - 공유 데이터 디렉터리에 같은 이름의 분류기가 이미 존재해야 합니다.
- `options["out"]`:
    - 파일 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - `.json` 확장자를 가져야 합니다.
### 6. Fit-predict
`fit_predict` 액션은 훈련 데이터를 사용하여 분류기를 훈련한 다음, 이를 사용하여 경성 및 연성 예측(확률)을 얻습니다.

#### 입력
|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | The name of the action. Must be `fit_predict`. | Yes |
| `name`      | `str`    | The name of the classifier to be used. | Yes |
| `X_train`   | `array` or `list` or `str` | The training data. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `y_train`   | `array` or `list` or `str` | The training target values. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `X_test`    | `array` or `list` or `str` | The test data. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `fit_params`| `dictionary`| Additional parameters to pass to the `fit` method of the classifier. | No |
| `options["out"]` | `str` | The output JSON filename to save the predictions in the shared data directory. If not provided, the predictions are returned in the job result. | No |

#### 사용법

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### 검증
- `name`:
    - 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - 공유 데이터 디렉터리에 같은 이름의 분류기가 이미 존재해야 합니다.

- `options["out"]`:
    - 파일 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - `.json` 확장자를 가져야 합니다.
### 7. Create-fit-predict
`create_fit_predict` 액션은 분류기를 생성하고, 제공된 훈련 데이터를 사용하여 훈련한 다음, 이를 사용하여 경성 및 연성 예측(확률)을 얻습니다.

#### 입력
| Name | Type | Description | Required |
|------|------|-------------|-----------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |
| `name` | `str` | The name of the classifier to be used. | Yes |
| `quantum_classifier` | `str` | The type of the classifier, i.e., `QuantumEnhancedEnsembleClassifier`. Default is `QuantumEnhancedEnsembleClassifier`. | No |
| `X_train` | `array` or `list` or `str` | The training data. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `y_train` | `array` or `list` or `str` | The training target values. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `X_test` | `array` or `list` or `str` | The test data. This can be a NumPy array, a list, or a string referencing a filename in the shared data directory. | Yes |
| `fit_params` | `dictionary` | Additional parameters to pass to the `fit` method of the classifier. | No |
| `options["save"]` | `boolean` | Whether to save to trained classifier in the shared data directory. Default is `True`. | No |
| `options["out"]` | `str` | The output JSON filename to save the predictions in the shared data directory. If not provided, the predictions are returned in the job result. | No |

#### 사용법

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### 검증
- `name`:
    - `options["save"]`가 `True`로 설정된 경우:
        - 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
        - 영숫자 문자와 밑줄만 포함할 수 있습니다.
        - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
        - 공유 데이터 디렉터리에 같은 이름의 분류기가 이미 존재하지 않아야 합니다.

- `options["out"]`:
    - 파일 이름은 고유해야 하며, 최대 64자 길이의 문자열이어야 합니다.
    - 영숫자 문자와 밑줄만 포함할 수 있습니다.
    - 문자로 시작해야 하며 밑줄로 끝날 수 없습니다.
    - `.json` 확장자를 가져야 합니다.
---
## 시작하기
[IBM Quantum Platform API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고, 다음과 같이 Qiskit Function을 선택합니다.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## 예시
이 예시에서는 "Singularity Machine Learning - Classification" 함수를 사용하여 두 개의 맞물린 반원 모양(moon-shaped)으로 이루어진 데이터셋을 분류합니다. 이 데이터셋은 합성 데이터이며, 2차원으로 구성되어 있고 이진 레이블로 표시되어 있습니다. 중심점 기반 클러스터링이나 선형 분류와 같은 알고리즘에 도전적인 과제가 되도록 설계되었습니다.
![Moons 데이터셋](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
이 과정을 통해 분류기를 생성하고, 훈련 데이터에 맞게 학습시키고, 테스트 데이터에 대한 예측에 활용하며, 완료 후 분류기를 삭제하는 방법을 알아볼 수 있습니다.
시작하기 전에 [scikit-learn](https://scikit-learn.org/)을 설치해야 합니다. 다음 명령어를 사용하여 설치하세요.